### 1. Carga de datos

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np

df = pd.read_parquet('/content/drive/MyDrive/TFM/comparacion_completa_v2.parquet')
print(df.shape)
print(df.columns.tolist())

Mounted at /content/drive
(242708, 26)
['store_nbr', 'item_nbr', 'time_idx', 'actual', 'v7_q0.02', 'v7_q0.1', 'v7_q0.25', 'v7_q0.5', 'v7_q0.75', 'v7_q0.9', 'v7_q0.98', 'v8_q0.02', 'v8_q0.1', 'v8_q0.25', 'v8_q0.5', 'v8_q0.75', 'v8_q0.9', 'v8_q0.98', 'v9_q0.02', 'v9_q0.1', 'v9_q0.25', 'v9_q0.5', 'v9_q0.75', 'v9_q0.9', 'v9_q0.98', 'pred_naive']


In [3]:
#Verificación previa de columnas
niveles_entrenados = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
cols_v9 = {q: f'v9_q{q}' for q in niveles_entrenados}

columnas_esperadas = list(cols_v9.values()) + ['actual', 'pred_naive', 'store_nbr', 'item_nbr']
faltantes = [c for c in columnas_esperadas if c not in df.columns]

if faltantes:
    print('FALTAN columnas esperadas:', faltantes)
    print('Columnas disponibles:', df.columns.tolist())
else:
    print('Todas las columnas esperadas están presentes.')
    print(df[columnas_esperadas].isna().sum())

Todas las columnas esperadas están presentes.
v9_q0.02      0
v9_q0.1       0
v9_q0.25      0
v9_q0.5       0
v9_q0.75      0
v9_q0.9       0
v9_q0.98      0
actual        0
pred_naive    0
store_nbr     0
item_nbr      0
dtype: int64


### 3. Cuantil interpolado para niveles no entrenados

0,25 coincide con el cuantil entrenado de DeepAR. 0,20 y 0,35 se calculan por interpolación lineal entre los dos cuantiles vecinos,

In [4]:
def cuantil_interpolado(df, nivel, columnas=cols_v9, niveles=niveles_entrenados):
    if nivel in niveles:
        return df[columnas[nivel]].values
    inferiores = [n for n in niveles if n < nivel]
    superiores = [n for n in niveles if n > nivel]
    if not inferiores or not superiores:
        raise ValueError(f'Nivel {nivel} fuera del rango entrenado ({min(niveles)}-{max(niveles)}), no se puede interpolar de forma confiable.')
    n_inf, n_sup = max(inferiores), min(superiores)
    y_inf, y_sup = df[columnas[n_inf]].values, df[columnas[n_sup]].values
    peso = (nivel - n_inf) / (n_sup - n_inf)
    return y_inf + peso * (y_sup - y_inf)

# prueba rápida: 0.25 debe coincidir exacto con la columna v9_q0.25
prueba = cuantil_interpolado(df, 0.25)
assert np.allclose(prueba, df['v9_q0.25'].values), 'El cuantil exacto no coincide — revisar función.'
print('Función de interpolación verificada sobre el nivel exacto 0,25.')

Función de interpolación verificada sobre el nivel exacto 0,25.


### 4. Costos por escenario en unidades relativas

In [5]:
escenarios = {
    0.20: {'Co': 1.0, 'Cu': 0.20 / (1 - 0.20)},
    0.25: {'Co': 1.0, 'Cu': 0.25 / (1 - 0.25)},
    0.35: {'Co': 1.0, 'Cu': 0.35 / (1 - 0.35)},
}
for m, c in escenarios.items():
    print(f"m={m}: Co={c['Co']:.3f}, Cu={c['Cu']:.3f} (razón crítica = {c['Cu']/(c['Cu']+c['Co']):.3f})")

m=0.2: Co=1.000, Cu=0.250 (razón crítica = 0.200)
m=0.25: Co=1.000, Cu=0.333 (razón crítica = 0.250)
m=0.35: Co=1.000, Cu=0.538 (razón crítica = 0.350)


### 5. Función de costo newsvendor

In [6]:
def costo_newsvendor(actual, Q, Cu, Co):
    diferencia = actual - Q
    return np.where(diferencia > 0, Cu * diferencia, Co * (-diferencia))

### 6. Tres políticas × tres escenarios

- **naive:** pedir según el baseline estacional (shift-7), sin ajustar por razón crítica.
- **deepar_mediana:** pedir siempre el cuantil 0,5 de DeepAR v9, ignorando la asimetría de costos.
- **deepar_newsvendor:** pedir el cuantil que corresponde a la razón crítica del escenario — la política propuesta.

In [7]:
df_valid = df.dropna(subset=['actual', 'pred_naive', 'v9_q0.5']).copy()
print(f'Filas usadas: {len(df_valid)} de {len(df)} (se descartan filas sin baseline naive, por el shift de 7 días al inicio de cada serie)')

resultados = []

for m, costos in escenarios.items():
    Cu, Co = costos['Cu'], costos['Co']
    Q_newsvendor = cuantil_interpolado(df_valid, m)

    politicas = {
        'naive': df_valid['pred_naive'].values,
        'deepar_mediana': df_valid['v9_q0.5'].values,
        'deepar_newsvendor': Q_newsvendor,
    }

    for nombre_politica, Q in politicas.items():
        costo_fila = costo_newsvendor(df_valid['actual'].values, Q, Cu, Co)

        tmp = df_valid[['store_nbr', 'item_nbr']].copy()
        tmp['costo'] = costo_fila
        por_serie = tmp.groupby(['store_nbr', 'item_nbr'])['costo'].mean()

        resultados.append({
            'escenario_m': m, 'Cu': round(Cu, 3), 'Co': Co, 'politica': nombre_politica,
            'costo_total': costo_fila.sum(),
            'costo_promedio_fila': costo_fila.mean(),
            'costo_promedio_serie_media': por_serie.mean(),
            'costo_promedio_serie_mediana': por_serie.median(),
            'n_filas': len(costo_fila),
        })

resultados_newsvendor = pd.DataFrame(resultados)
resultados_newsvendor

Filas usadas: 242708 de 242708 (se descartan filas sin baseline naive, por el shift de 7 días al inicio de cada serie)


,escenario_m,Cu,Co,politica,costo_total,costo_promedio_fila,costo_promedio_serie_media,costo_promedio_serie_mediana,n_filas
0,0.20,0.250,1.0,naive,528379.00000,2.177015,2.176523,1.766393,242708
1,0.20,0.250,1.0,deepar_mediana,376980.75000,1.553228,1.552881,1.122951,242708
2,0.20,0.250,1.0,deepar_newsvendor,229437.09375,0.945322,0.945021,0.718169,242708
3,0.25,0.333,1.0,naive,561177.00000,2.312149,2.311615,1.885246,242708
4,0.25,0.333,1.0,deepar_mediana,402244.34375,1.657318,1.656930,1.221311,242708
5,0.25,0.333,1.0,deepar_newsvendor,283800.25000,1.169307,1.168943,0.886612,242708
6,0.35,0.538,1.0,naive,641910.56250,2.644785,2.644148,2.170239,242708
7,0.35,0.538,1.0,deepar_mediana,464431.59375,1.913540,1.913051,1.456494,242708
8,0.35,0.538,1.0,deepar_newsvendor,404506.37500,1.666638,1.666138,1.288588,242708


### 7. Tabla resumen y % de reducción de costo

In [8]:
tabla = resultados_newsvendor.pivot(index='escenario_m', columns='politica', values='costo_promedio_fila')
tabla['reduccion_mediana_vs_naive_%'] = (1 - tabla['deepar_mediana'] / tabla['naive']) * 100
tabla['reduccion_newsvendor_vs_naive_%'] = (1 - tabla['deepar_newsvendor'] / tabla['naive']) * 100
tabla['reduccion_newsvendor_vs_mediana_%'] = (1 - tabla['deepar_newsvendor'] / tabla['deepar_mediana']) * 100

print(tabla.to_string())

resultados_newsvendor.to_csv('/content/drive/MyDrive/TFM/resultados_newsvendor.csv', index=False)
tabla.to_csv('/content/drive/MyDrive/TFM/resultados_newsvendor_resumen.csv')

politica     deepar_mediana  deepar_newsvendor     naive  reduccion_mediana_vs_naive_%  reduccion_newsvendor_vs_naive_%  reduccion_newsvendor_vs_mediana_%
escenario_m                                                                                                                                               
0.20               1.553228           0.945322  2.177015                     28.653341                        56.577171                          39.138245
0.25               1.657318           1.169307  2.312149                     28.321308                        49.427677                          29.445808
0.35               1.913540           1.666638  2.644785                     27.648556                        36.983997                          12.902916
